# Calculating Sequence Features

Computes sequence features for every amino-acid sequence in a CSV:

| Feature | Definition |
|---|---|
| `A` ... `Y` | Count of each of the 20 amino acids |
| `Aromatics` | W + F + Y count |
| `WFYLM` | W + F + Y + L + M count |
| `Net_Charge` | NCPR × sequence length (localCIDER) |
| `HydrophobicityKD` | Normalized Kyte-Doolittle hydropathy, `get_uversky_hydropathy()` (localCIDER) |
| `HydrophobicityWW` | Wimley-White hydropathy, `get_WW_hydropathy()` (localCIDER) |
| `Kappa` | Patterning of oppositely charged residues, κ (Das & Pappu 2013) |
| `Omega` | Patterning of charged residues + prolines vs. all other residues, Ω (Martin et al. 2016) |
| `OmegaWFYLM_DE` | Ω-style patterning of {W,F,Y,L,M} vs. {D,E}, `get_kappa_X(['D','E'], ['W','F','Y','L','M'])` |
| `OmegaWF_D` | Ω-style patterning of {W,F} vs. {D}, `get_kappa_X(['D'], ['W','F'])` |

**Requirements:** `pandas`, `numpy`, `tqdm`, `localcider`.
**Input:** a CSV with an amino-acid sequence column (default `AAseq`).

In [1]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from localcider.sequenceParameters import SequenceParameters

In [2]:
INPUT_CSV = "/mnt/c/Users/music/OneDrive/Desktop/CompLab/SynAD/data/500k_AA_SS.csv"            # path to input CSV
OUTPUT_CSV = "/mnt/c/Users/music/Downloads/500k_AA_SS_seqfeats.csv"  # path to write results
AA_COL = "AAseq"                       # column holding amino-acid sequences

df = pd.read_csv(INPUT_CSV)
seqs = df[AA_COL].fillna("").astype(str).str.strip().str.upper()

n_empty = (seqs == "").sum()
if n_empty:
    print(f"Warning: {n_empty} rows have an empty {AA_COL}; their features will be NaN")
print(f"{len(df)} sequences loaded")
df.head()

500000 sequences loaded


,AAseq,State Size
0,PPDEAVINYGTAQGLDKWEEEMMKDEAEPCGEKYTSIGTD,1
1,SVQEEMDDGVPGGFASPYLALDQPVQSFEEKTRSFMPSDS,1
2,LEFYYWFPMEGPTHFQKDTTFSFMDEVNNGDELSDIGNST,1
3,RLVGDIFLLQNPTEGEILKLDNQAMDEDRFTSVQVEEWLD,1
4,TLGCLAEDGHHALPSAPILEQPAATCSLLGYYLAKASPSN,1


In [3]:
AALIST = list("ACDEFGHIKLMNPQRSTVWY")

counts = pd.DataFrame({aa: seqs.str.count(aa) for aa in AALIST}, index=df.index)

comp = pd.DataFrame(index=df.index)
comp["Aromatics"] = counts[["W", "F", "Y"]].sum(axis=1)
comp["WFYLM"] = counts[["W", "F", "Y", "L", "M"]].sum(axis=1)

In [4]:
CIDER_FEATURES = ["Net_Charge", "HydrophobicityKD", "HydrophobicityWW",
                  "Kappa", "Omega", "OmegaWFYLM_DE", "OmegaWF_D"]

def cider_features(seq):
    """Return a dict of localCIDER features for one sequence (NaN on failure)."""
    if not seq:
        return {f: np.nan for f in CIDER_FEATURES}
    sp = SequenceParameters(seq)
    return {
        "Net_Charge":       sp.get_NCPR() * len(seq),
        "HydrophobicityKD": sp.get_uversky_hydropathy(),
        "HydrophobicityWW": sp.get_WW_hydropathy(),
        "Kappa":            sp.get_kappa(),
        "Omega":            sp.get_Omega(),
        "OmegaWFYLM_DE":    sp.get_kappa_X(["D", "E"], ["W", "F", "Y", "L", "M"]),
        "OmegaWF_D":        sp.get_kappa_X(["D"], ["W", "F"]),
    }

rows, errors = [], {}
for idx, seq in tqdm(seqs.items(), total=len(seqs), desc="localCIDER"):
    try:
        rows.append(cider_features(seq))
    except Exception as e:
        rows.append({f: np.nan for f in CIDER_FEATURES})
        errors[idx] = str(e)

cider = pd.DataFrame(rows, index=df.index)
if errors:
    print(f"Warning: {len(errors)} sequences failed; examples: {dict(list(errors.items())[:5])}")

localCIDER:   0%|          | 0/500000 [00:00<?, ?it/s]

In [5]:
out = pd.concat([df, cider[["Net_Charge"]], comp,
                 cider.drop(columns="Net_Charge"), counts], axis=1)
out.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {out.shape[0]} rows x {out.shape[1]} columns to {OUTPUT_CSV}")
out.head()

Wrote 500000 rows x 31 columns to /mnt/c/Users/music/Downloads/500k_AA_SS_seqfeats.csv


,AAseq,State Size,Net_Charge,Aromatics,WFYLM,HydrophobicityKD,HydrophobicityWW,Kappa,Omega,OmegaWFYLM_DE,...,M,N,P,Q,R,S,T,V,W,Y
0,PPDEAVINYGTAQGLDKWEEEMMKDEAEPCGEKYTSIGTD,1,-8.0,3,6,0.385278,-0.49850,0.136843,0.292284,0.117811,...,2,1,3,1,0,1,3,1,1,2
1,SVQEEMDDGVPGGFASPYLALDQPVQSFEEKTRSFMPSDS,1,-6.0,4,8,0.429722,-0.34825,0.306337,0.066155,0.169576,...,2,0,4,3,1,6,1,3,0,1
2,LEFYYWFPMEGPTHFQKDTTFSFMDEVNNGDELSDIGNST,1,-7.0,8,12,0.421944,-0.18675,0.200756,0.123285,0.192691,...,2,3,2,1,0,3,4,1,1,2
3,RLVGDIFLLQNPTEGEILKLDNQAMDEDRFTSVQVEEWLD,1,-7.0,3,10,0.452222,-0.35950,0.158182,0.137986,0.091878,...,1,2,1,3,2,1,2,3,1,0
4,TLGCLAEDGHHALPSAPILEQPAATCSLLGYYLAKASPSN,1,-2.0,2,9,0.516944,-0.16025,0.617010,0.397816,0.370253,...,0,1,4,1,0,4,2,0,0,2
